In [1]:
# ============================================================
# CELL 1: Environment Setup & Master YAML Configuration
# ============================================================
import os
import shutil
import yaml

!pip install -q ultralytics ncnn qai_hub pnnx

base_dir = '/kaggle/working/disaster_dataset'
dirs = ['images/train', 'images/val', 'labels/train', 'labels/val']

shutil.rmtree(base_dir, ignore_errors=True)
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)

# Master 7-Class Ontology
classes_config = {
    'path': base_dir,
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'person',
        1: 'fire',
        2: 'smoke',
        3: 'floodwater',
        4: 'structural_damage',
        5: 'landslide',
        6: 'exposed_wire'
    }
}

yaml_path = '/kaggle/working/classes.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(classes_config, f, sort_keys=False)

print(f"Workspace ready. Master config written to: {yaml_path}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
boto3 1.43.36 requires s3transfer<0.20.0,>=0.19.0, but you have s3transfer 0.13.1 which is incompatible.
Workspace ready. Master config written to: /kaggle/working/classes.yaml


In [2]:
# ============================================================
# CELL 2: Pre-Processing Converters (Masks & XML to YOLO)
# ============================================================
import cv2
import glob
import numpy as np
import xml.etree.ElementTree as ET

def sanity_check_box(box):
    cls_id, xc, yc, w, h = box
    if w < 0.0125 or h < 0.0125 or (w * h > 0.90): return None
    xc, yc = max(0.0001, min(0.9999, float(xc))), max(0.0001, min(0.9999, float(yc)))
    w, h = max(0.0001, min(0.9999, float(w))), max(0.0001, min(0.9999, float(h)))
    return [int(cls_id), xc, yc, w, h]

def convert_masks_to_boxes(mask_dir, output_label_dir, target_class_id):
    os.makedirs(output_label_dir, exist_ok=True)
    for mask_path in glob.glob(os.path.join(mask_dir, "*.png")):
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None: continue
        h_img, w_img = mask.shape
        _, thresh = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_boxes = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            raw = [target_class_id, (x+w/2.0)/w_img, (y+h/2.0)/h_img, w/float(w_img), h/float(h_img)]
            cleaned = sanity_check_box(raw)
            if cleaned: valid_boxes.append(cleaned)
        if valid_boxes:
            with open(os.path.join(output_label_dir, os.path.basename(mask_path).replace('.png', '.txt')), 'w') as f:
                for b in valid_boxes: f.write(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")

def convert_voc_xml(xml_dir, output_label_dir, target_class_id):
    os.makedirs(output_label_dir, exist_ok=True)
    for xml_path in glob.glob(os.path.join(xml_dir, "*.xml")):
        try:
            tree = ET.parse(xml_path)
            w_img = int(tree.getroot().find('size/width').text)
            h_img = int(tree.getroot().find('size/height').text)
            valid_boxes = []
            for obj in tree.getroot().findall('object'):
                bnd = obj.find('bndbox')
                xmin, ymin, xmax, ymax = float(bnd.find('xmin').text), float(bnd.find('ymin').text), float(bnd.find('xmax').text), float(bnd.find('ymax').text)
                raw = [target_class_id, ((xmin+xmax)/2.0)/w_img, ((ymin+ymax)/2.0)/h_img, (xmax-xmin)/w_img, (ymax-ymin)/h_img]
                cleaned = sanity_check_box(raw)
                if cleaned: valid_boxes.append(cleaned)
            if valid_boxes:
                with open(os.path.join(output_label_dir, os.path.basename(xml_path).replace('.xml', '.txt')), 'w') as f:
                    for b in valid_boxes: f.write(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")
        except Exception: continue
print("Conversion functions loaded.")

Conversion functions loaded.


In [3]:
# ============================================================
# CELL 2.5: The Format Rescuer (TTPLA JSON + RescueNet Pixel-Index)
# ============================================================
import os
import glob
import json
import cv2
import numpy as np
from PIL import Image

TEMP_LBL_DIR = '/kaggle/working/rescued_labels'
os.makedirs(TEMP_LBL_DIR, exist_ok=True)

print("1. Rescuing TTPLA Wires (JSON to TXT)...")
wire_count = 0
for json_path in glob.glob('/kaggle/input/datasets/giorgioubbriaco/**/*.json', recursive=True):
    with open(json_path, 'r') as f:
        data = json.load(f)
        img_dict = {img['id']: img for img in data['images']}
        for ann in data['annotations']:
            if ann['category_id'] not in [1, 2]: continue
            img_id = ann['image_id']
            if img_id not in img_dict: continue
            img_info = img_dict[img_id]
            w_img, h_img = float(img_info['width']), float(img_info['height'])
            x, y, w, h = ann['bbox']
            xc, yc = (x + w/2.0) / w_img, (y + h/2.0) / h_img
            wn, hn = w / w_img, h / h_img
            txt_name = os.path.basename(img_info['file_name']).replace('.jpg', '.txt')
            with open(os.path.join(TEMP_LBL_DIR, txt_name), 'a') as f_out:
                f_out.write(f"6 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")
            wire_count += 1
print(f"   -> Rescued {wire_count} Wire bounding boxes.")

print("2. Rescuing Floodwater (RescueNet Pixel Value 2 to TXT)...")
water_count = 0
for mask_path in glob.glob('/kaggle/input/datasets/yaroslavchyrko/**/*.png', recursive=True):
    try:
        mask_img = Image.open(mask_path)
        mask = np.array(mask_img)
    except Exception:
        continue
    water_mask = np.where(mask == 2, 255, 0).astype(np.uint8)
    contours, _ = cv2.findContours(water_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h_img, w_img = mask.shape
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w < 5 or h < 5: continue
        xc, yc = (x + w/2.0) / w_img, (y + h/2.0) / h_img
        wn, hn = w / w_img, h / h_img
        boxes.append(f"3 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")
    if boxes:
        stem = os.path.splitext(os.path.basename(mask_path))[0]
        stem = stem.replace('-label', '').replace('_label', '').replace('_mask', '')
        with open(os.path.join(TEMP_LBL_DIR, f"{stem}.txt"), 'w') as f_out:
            f_out.writelines(boxes)
        water_count += 1

print(f"   -> Rescued {water_count} Floodwater images.")
print(f"Rescue complete! Missing labels staged in {TEMP_LBL_DIR}")

1. Rescuing TTPLA Wires (JSON to TXT)...
   -> Rescued 0 Wire bounding boxes.
2. Rescuing Floodwater (RescueNet Pixel Value 2 to TXT)...
   -> Rescued 0 Floodwater images.
Rescue complete! Missing labels staged in /kaggle/working/rescued_labels


In [4]:
# ============================================================
# CELL 3: Unified Dataset Migration (V2 Production)
# ============================================================
import os
import shutil
import random
import cv2
import glob

BASE_DIR = '/kaggle/working/disaster_dataset'
TRAIN_IMG_DIR, TRAIN_LBL_DIR = os.path.join(BASE_DIR, 'images/train'), os.path.join(BASE_DIR, 'labels/train')
VAL_IMG_DIR, VAL_LBL_DIR = os.path.join(BASE_DIR, 'images/val'), os.path.join(BASE_DIR, 'labels/val')

shutil.rmtree(BASE_DIR, ignore_errors=True)
for d in [TRAIN_IMG_DIR, TRAIN_LBL_DIR, VAL_IMG_DIR, VAL_LBL_DIR]:
    os.makedirs(d, exist_ok=True)

# Master Class Ontology — per-source remapping into the 7 unified classes
DATASET_CLASS_MAPS = {
    'sayedgamal99': {0: 2, 1: 1},         # Smoke-Fire: smoke->2, fire->1
    'kushagrapandya': {1: 0, 2: 0},       # VisDrone: pedestrian/person->0
    'horacelo': {0: 5},                   # Landslide
    'anamibnjafar0': {0: 1},              # FlameVision: fire
    'giorgioubbriaco': {0: 6, 1: 6},      # TTPLA: wires
    'yaroslavchyrko': {0: 3},             # RescueNet: floodwater
    'aletbm': {0: 4},                     # FloodNet: structural damage
    'nikolasgegenava': {0: 4},            # SARD: structural damage
}

print("Indexing all images across input datasets...")
image_pool = {}
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp'}

for root, _, files in os.walk('/kaggle/input/datasets'):
    if 'awsaf49' in root: continue  # COCO excluded — pretrained weights only
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        if ext in valid_exts and not f.startswith('.'):
            stem = os.path.splitext(f)[0]
            image_pool[stem] = os.path.join(root, f)

print(f"Total image pool indexed: {len(image_pool)} files.")

raw_records = []
search_paths = ['/kaggle/input/datasets', '/kaggle/working/rescued_labels']

for search_dir in search_paths:
    for root, _, files in os.walk(search_dir):
        if 'awsaf49' in root: continue
        for f in files:
            if f.endswith('.txt') and not any(f.lower().endswith(x) for x in ['classes.txt', 'readme.txt', 'note-v1.0.txt']):
                stem = os.path.splitext(f)[0]
                if stem in image_pool:
                    lbl_path = os.path.join(root, f)
                    if os.path.getsize(lbl_path) > 0:
                        target_img = image_pool[stem]
                        img_root = os.path.dirname(target_img).lower()
                        raw_records.append((lbl_path, target_img, f, os.path.basename(target_img), img_root))

random.seed(42)
random.shuffle(raw_records)
split_idx = int(len(raw_records) * 0.8)
train_records, val_records = raw_records[:split_idx], raw_records[split_idx:]

def process_and_write(records, target_lbl_dir, target_img_dir, prefix):
    m_count = 0
    for idx, (lbl_src, img_src, lbl_name, img_name, root_path) in enumerate(records):
        with open(lbl_src, 'r') as f_in:
            lines = f_in.readlines()

        matched_map = next((mapping for key, mapping in DATASET_CLASS_MAPS.items() if key in root_path), None)
        clean_lines = []
        img_w, img_h = None, None

        for line in lines:
            parts = line.strip().replace(',', ' ').split()
            if len(parts) < 5: continue
            if not parts[0].lstrip('-').replace('.','',1).isdigit(): continue

            try:
                if len(parts) >= 8 and 'kushagrapandya' in root_path:
                    # VisDrone 10-column format
                    src_cls = int(parts[5])
                    xmin, ymin, box_w, box_h = map(float, parts[0:4])
                    xc, yc = xmin + (box_w / 2.0), ymin + (box_h / 2.0)
                    w, h = box_w, box_h
                else:
                    src_cls = int(float(parts[0]))
                    xc, yc, w, h = map(float, parts[1:5])
            except (ValueError, IndexError):
                continue

            # Bypass remapping for rescued labels (already in master IDs)
            if 'rescued_labels' in lbl_src:
                target_cls = src_cls
            elif matched_map:
                if src_cls in matched_map: target_cls = matched_map[src_cls]
                else: continue
            else:
                if src_cls in range(7): target_cls = src_cls
                else: continue

            if w > 1.0 or h > 1.0 or xc > 1.0:
                if img_w is None:
                    img = cv2.imread(img_src)
                    if img is None: break
                    img_h, img_w = img.shape[:2]
                xc /= float(img_w); yc /= float(img_h)
                w /= float(img_w); h /= float(img_h)

            w, h = max(0.0001, min(1.0, w)), max(0.0001, min(1.0, h))
            xc, yc = max(0.0001, min(0.9999, xc)), max(0.0001, min(0.9999, yc))
            if w < 0.001 or h < 0.001: continue

            clean_lines.append(f"{target_cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

        if not clean_lines: continue

        safe_lbl = f"{prefix}_{idx}_{lbl_name}"
        safe_img = f"{prefix}_{idx}_{img_name}"
        with open(os.path.join(target_lbl_dir, safe_lbl), 'w') as f_out:
            f_out.writelines(clean_lines)

        target_img_path = os.path.join(target_img_dir, safe_img)
        if not os.path.exists(target_img_path):
            os.symlink(img_src, target_img_path)
        m_count += 1

    return m_count

train_count = process_and_write(train_records, TRAIN_LBL_DIR, TRAIN_IMG_DIR, "tr")
val_count = process_and_write(val_records, VAL_LBL_DIR, VAL_IMG_DIR, "val")

for c in glob.glob('/kaggle/working/disaster_dataset/labels/*/*.cache'):
    os.remove(c)

print(f"\nMigration Complete! Processed: Train={train_count} | Val={val_count}")

Indexing all images across input datasets...
Total image pool indexed: 33832 files.

Migration Complete! Processed: Train=26257 | Val=6580


In [5]:
# ============================================================
# CELL 3.5: Pre-Training Gatekeeper (< 5 seconds)
# ============================================================
import os
import glob
from collections import Counter

class_names = {
    0: 'person', 1: 'fire', 2: 'smoke', 3: 'floodwater',
    4: 'structural_damage', 5: 'landslide', 6: 'exposed_wire'
}

def scan_split(split_name):
    label_files = glob.glob(f'/kaggle/working/disaster_dataset/labels/{split_name}/*.txt')
    counts = Counter()
    total_boxes = 0
    for f in label_files:
        with open(f, 'r') as fp:
            for line in fp:
                p = line.strip().split()
                if len(p) >= 5:
                    try:
                        counts[int(p[0])] += 1
                        total_boxes += 1
                    except ValueError:
                        pass
    return counts, len(label_files), total_boxes

tr_counts, tr_imgs, tr_boxes = scan_split('train')
val_counts, val_imgs, val_boxes = scan_split('val')

print("================ DATASET PRE-FLIGHT AUDIT ================")
print(f"Images : Train = {tr_imgs:<6} | Val = {val_imgs}")
print(f"Boxes  : Train = {tr_boxes:<6} | Val = {val_boxes}")
print("-" * 58)
print(f"{'Class ID':<8} | {'Class Name':<18} | {'Train Boxes':<12} | {'Val Boxes'}")
print("-" * 58)

zero_classes = []
for cid, name in class_names.items():
    t_cnt = tr_counts.get(cid, 0)
    v_cnt = val_counts.get(cid, 0)
    if t_cnt == 0 or v_cnt == 0:
        zero_classes.append(name)
    print(f"{cid:<8} | {name:<18} | {t_cnt:<12} | {v_cnt}")
print("=" * 58)

if zero_classes:
    raise AssertionError(
        f"ABORT TRAINING: Zero instances found for classes: {zero_classes}!\n"
        f"Do not proceed to Phase 1 until the parser captures all classes."
    )
else:
    print("✅ PRE-FLIGHT PASSED: All 7 disaster classes verified. Safe to run Phase 1.")

================ DATASET PRE-FLIGHT AUDIT ================
Images : Train = 26257  | Val = 6580
Boxes  : Train = 120388 | Val = 27869
----------------------------------------------------------
Class ID | Class Name         | Train Boxes  | Val Boxes
----------------------------------------------------------
0        | person             | 80776        | 17674
1        | fire               | 14322        | 3582
2        | smoke              | 11355        | 2908
3        | floodwater         | 7131         | 1976
4        | structural_damage  | 5924         | 1500
5        | landslide          | 389          | 86
6        | exposed_wire       | 491          | 143
✅ PRE-FLIGHT PASSED: All 7 disaster classes verified. Safe to run Phase 1.


In [6]:
# ============================================================
# CELL 4: Phase 1 Training (YOLOv26n - Frozen Backbone)
# Head-focused warmup: backbone frozen (first 10 layers),
# higher LR to adapt the detection head to the 7-class ontology.
# ============================================================
from ultralytics import YOLO

print("Starting Phase 1 with YOLOv26n (frozen backbone)...")
model_p1 = YOLO('yolo26n.pt')

model_p1.train(
    data='/kaggle/working/classes.yaml',
    epochs=15,
    imgsz=640,
    batch=16,
    workers=2,            # Prevents Kaggle vCPU thread locks
    freeze=10,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    rect=False,

    # Safe scaling without per-batch CPU stall
    multi_scale=False,
    scale=0.5,
    mosaic=1.0,

    # Hallucination fixers (disabled)
    copy_paste=0.0,
    mixup=0.0,

    close_mosaic=10,
    box=7.5,
    cls=0.5,
    max_det=900,
    cache='ram',
    device=0,
    project='/kaggle/working/models',
    name='v2_phase1_frozen'
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Starting Phase 1 with YOLOv26n (frozen backbone)...
Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/classes.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/15      4.57G      2.367      2.598    0.01308         57        640: 100% ━━━━━━━━━━━━ 1638/1638 5.0it/s 5:29
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 206/206 5.4it/s 37.9s
                   all       6565      27850      0.614      0.347      0.375      0.179

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/15      4.57G      2.329      2.397    0.01253         61        640: 100% ━━━━━━━━━━━━ 1638/1638 4.8it/s 5:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 206/206 5.4it/s 38.0s
                   all       6565      27850        0.5      0.414       0.41      0.205

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/15      4.57G      2.286      2.294    0.01214        218        640: 100% ━━━━━━━━━━━━ 1638/1638 4.8it/s 5:43
                 Class     Images  Instances

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c967d5fb2c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

In [7]:
# ============================================================
# CELL 5: Phase 2 Training (Full Fine-Tune + Multi-Scale)
# Entire network unfrozen, low LR, multi-scale input for
# robustness to victims/hazards at varying drone altitudes.
# ============================================================
from ultralytics import YOLO
import glob, os

p1_weights = max(glob.glob('/kaggle/working/models/v2_phase1_frozen*/weights/best.pt'), key=os.path.getctime)
model_p2 = YOLO(p1_weights)

print("Starting Phase 2 with Multi-Scale (fully unfrozen)...")
model_p2.train(
    data='/kaggle/working/classes.yaml',
    epochs=40,
    imgsz=640,
    batch=16,
    workers=2,
    freeze=0,
    optimizer='AdamW',
    lr0=0.0001,
    lrf=0.01,
    rect=False,

    multi_scale=True,     # Scale jitter for altitude/distance robustness
    scale=0.5,
    mosaic=1.0,

    copy_paste=0.0,
    mixup=0.0,

    close_mosaic=10,
    box=7.5,
    cls=0.5,
    max_det=900,
    cache='ram',
    device=0,
    project='/kaggle/working/models',
    name='v2_phase2_final'
)

Starting Phase 2 with Multi-Scale (fully unfrozen)...
Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/classes.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=900, mixup=0.0, mode=train, model=/kaggle/working/models/v2_phase1_frozen/weights/best.pt, m

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c965c86a780>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

In [8]:
# ============================================================
# CELL 6: Extract Best Weights
# ============================================================
import shutil

p2_weights = max(glob.glob('/kaggle/working/models/v2_phase2_final*/weights/best.pt'), key=os.path.getctime)
shutil.copy(p2_weights, '/kaggle/working/best.pt')

best_model = YOLO('/kaggle/working/best.pt')
print("Best weights extracted:", best_model.model_name,
      "| size:", os.path.getsize('/kaggle/working/best.pt') / 1e6, "MB")

Best weights extracted: /kaggle/working/best.pt | size: 5.366725 MB


In [9]:
# ============================================================
# CELL 7: ONNX Export (Qualcomm NPU deployment path)
# ============================================================
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True)
print("ONNX exported to:", onnx_path, "| size:", os.path.getsize(onnx_path) / 1e6, "MB")

Ultralytics 8.4.161 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 120 layers, 2,376,201 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/kaggle/working/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 11, 8400) (5.1 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 605ms
 Downloaded onnxruntime
Prepared 2 packages in 3.90s
Installed 2 packages in 27ms
 + onnxruntime==1.30.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 7.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 11.0s, s